# Flow direction data

The `FlwdirRaster` object is at the core of pyflwdir. It stores gridded flow-direction data in a common, actionable format, where each cell is represented by the linear index of its next downstream cell.

We currently support two local flow-direction (D8) formats: ArcGIS **D8** and PCRaster **LDD** (see figure). We also support one global flow-direction format: CaMa-Flood **NEXTXY**. Local formats describe the downstream cell by its relative direction to a neighboring cell, while the global format describes it by row and column indices.

<!-- ![](../docs/_static/ftypes.png "Local flow direction types") -->
<img src="https://raw.githubusercontent.com/Deltares/pyflwdir/main/docs/_static/ftypes.png" width="500"/>

We read the flow direction raster data, including meta-data, using [rasterio](https://rasterio.readthedocs.io/en/latest/)

In [ ]:
import rasterio

with rasterio.open("rhine_d8.tif", "r") as src:
    flwdir = src.read(1)
    transform = src.transform
    crs = src.crs
    latlon = crs.to_epsg() == 4326

Next, we parse this data to a `FlwdirRaster` object, the core object 
to work with flow direction data. In this step the D8 data is parsed to an actionable format.

<div class="alert alert-info">

NOTE: The first call to most methods may be slow because Numba compiles the code just in time. Subsequent calls to the same method, even with different arguments, are usually much faster!
    
</div>

In [ ]:
import pyflwdir

flw = pyflwdir.from_array(
    flwdir, ftype="d8", transform=transform, latlon=latlon, cache=True
)

In [ ]:
# When printing the FlwdirRaster instance we see its attributes.
print(flw)

We can then use the many methods available on the `FlwdirRaster` object. See the [FlwdirRaster API](https://deltares.github.io/pyflwdir/latest/_generated/pyflwdir.FlwdirRaster.html).

To visualize the flow directions, we derive a vector stream network using the [streams()](https://deltares.github.io/pyflwdir/latest/_generated/pyflwdir.FlwdirRaster.streams.html) method. Each line represents a stream segment with a Strahler order of at least `min_sto`, as computed by [stream_order()](https://deltares.github.io/pyflwdir/latest/_generated/pyflwdir.FlwdirRaster.stream_order.html). The line features are converted to a [GeoDataFrame](https://geopandas.org/data_structures.html#geodataframe) for visualization.

In [ ]:
import geopandas as gpd

feats = flw.streams(min_sto=4)
gdf = gpd.GeoDataFrame.from_features(feats, crs=crs)
gdf.head()

In [ ]:
import numpy as np
from matplotlib import cm, colors

# local convenience methods (see utils.py script in notebooks folder)
from utils import quickplot  # data specific quick plot method

# keyword arguments passed to GeoDataFrame.plot()
gdf_plot_kwds = {
    "column": "strord",
    "cmap": colors.ListedColormap(cm.Blues(np.linspace(0.4, 1, 7))),
}
# plot streams with hillshade from elevation data (see utils.py)
ax = quickplot(gdfs=[(gdf, gdf_plot_kwds)], title="Streams")